In [47]:
import pandas as pd
import numpy as np

In [48]:
from lightgbm import LGBMClassifier

In [49]:
from sklearn.metrics import recall_score,precision_score,confusion_matrix,f1_score,roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

In [50]:
X_train = pd.read_csv(r'D:\churn prediction\data\interim\X_train.csv')
y_train = pd.read_csv(r'D:\churn prediction\data\interim\y_train.csv')

X_test = pd.read_csv(r'D:\churn prediction\data\interim\X_test.csv')
y_test = pd.read_csv(r'D:\churn prediction\data\interim\y_test.csv')



In [51]:
model1 = LogisticRegression(random_state=42)

model2 = DecisionTreeClassifier(random_state=42)

model3 = RandomForestClassifier(random_state=42)

model4 = GradientBoostingClassifier(random_state=42)

model5 = AdaBoostClassifier(random_state=42)

model6 = XGBClassifier(random_state=42)

model_7 = LGBMClassifier(
    random_state=42,
    verbose=-1
)

In [52]:
from src.components.module_03_data_transformation import DataTransformation

transformation = DataTransformation()

num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

tnf = transformation.build(num_cols,cat_cols)



2026-06-02 20:07:30,115 - module_03_data_transformation.py - INFO - pipeline built num:14 cat:1


In [53]:
from sklearn.model_selection import cross_val_score

cv_auc_scores = []

model_list = [model1,model2,model3,model4,model5,model6,model_7]

recall_train_list = []
recall_test_list = []


precision_train_list = []
precision_test_list = []

auc_score_train_list = []
auc_score_test_list = []

f1_score_train = []
f1_score_test = []

model_names = []
for model in model_list:
    model_pipe = Pipeline([
        ('tnf',tnf),
        (f'{model.__class__.__name__}',model)
    ])
    
    cv_score = cross_val_score(
        model_pipe,
        X_train,
        y_train.values.ravel(),
        cv=5,
        scoring='roc_auc'
    )
    
    cv_auc_scores.append(cv_score.mean())
    
    #train model on data
    model_pipe.fit(X_train,y_train.values.ravel())
    model_names.append(model_pipe[1].__class__.__name__ )
    
    #predictions of model on train and test both
    test_pred = model_pipe.predict(X_test)
    train_pred = model_pipe.predict(X_train)
    
    # probablity
    prob_train = model_pipe.predict_proba(X_train)[:,1]
    prob_test = model_pipe.predict_proba(X_test)[:,1]
    

    #recall score calculation of test and train
    recall_train_list.append(recall_score(y_train,train_pred))
    recall_test_list.append(recall_score(y_test,test_pred))
    

    #precision score calculation of test and train
    precision_train_list.append(precision_score(y_train,train_pred)) 
    precision_test_list.append(precision_score(y_test,test_pred))
    
    f1_score_test.append(f1_score(y_test,test_pred))
    f1_score_train.append(f1_score(y_train,train_pred))
    
    auc_score_test_list.append(roc_auc_score(y_test,prob_test))
    auc_score_train_list.append(roc_auc_score(y_train,prob_train))
    
    

d:\churn prediction\venv_mlflow\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\churn prediction\venv_mlflow\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\churn prediction\venv_mlflow\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\churn prediction\venv_mlflow\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\churn prediction\venv_mlflow\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [56]:
results = pd.DataFrame({
    'Model': model_names,
    'CV_AUC': cv_auc_scores,
    'Train_AUC': auc_score_train_list,
    'Test_AUC': auc_score_test_list,
    'Train_F1': f1_score_train,
    'Test_F1': f1_score_test,
    'Train_Recall': recall_train_list,
    'Test_Recall': recall_test_list,
    'Train_Precision': precision_train_list,
    'Test_Precision': precision_test_list
})

results.sort_values(by=['CV_AUC'],ascending=False)

,Model,CV_AUC,Train_AUC,Test_AUC,Train_F1,Test_F1,Train_Recall,Test_Recall,Train_Precision,Test_Precision
6,LGBMClassifier,0.927614,1.000000,0.927954,1.000000,0.898876,1.000000,0.842105,1.000000,0.963855
5,XGBClassifier,0.923623,1.000000,0.932886,1.000000,0.900000,1.000000,0.852632,1.000000,0.952941
2,RandomForestClassifier,0.923230,1.000000,0.934008,0.998710,0.880000,0.997423,0.810526,1.000000,0.962500
3,GradientBoostingClassifier,0.920225,0.989311,0.923463,0.934066,0.901099,0.876289,0.863158,1.000000,0.942529
1,DecisionTreeClassifier,0.895422,1.000000,0.910600,1.000000,0.815920,1.000000,0.863158,1.000000,0.773585
4,AdaBoostClassifier,0.894650,0.911308,0.913462,0.633914,0.728324,0.515464,0.663158,0.823045,0.807692
0,LogisticRegression,0.809303,0.838095,0.813563,0.383363,0.302158,0.273196,0.221053,0.642424,0.477273


* top 3 models:
* LGBM, XGB and Rf are chosen for further hyperparameter since they have more stable auc for train,